# 🚦 Traffic Demand Prediction
*An Elite Ensemble Model (LightGBM + XGBoost) featuring Cyclical Time, Smoothed Target Encoding, and Peak-Demand Weighting.*

In [ ]:
%pip install optuna
%pip install scikit-learn
%pip install lightgbm

In [1]:
import os
import multiprocessing
# Prevents joblib crashes on Windows
os.environ['LOKY_MAX_CPU_COUNT'] = str(multiprocessing.cpu_count() or 4)

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import VotingRegressor
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
print("Environment Ready for Final Submission Generation!")

Environment Ready for Final Submission Generation!


In [2]:
def generate_ultimate_submission():
    print("Loading Train and Test datasets...")
    train = pd.read_csv('C:/Users/tanay/Documents/gridlock/e88186124ec611f1/dataset/train.csv')
    test = pd.read_csv('C:/Users/tanay/Documents/gridlock/e88186124ec611f1/dataset/test.csv')
    
    train_len = len(train)
    test_indices = test['Index'].copy()
    
    print("Applying Winning Feature Engineering...")
    # --- 1. Smoothed Target Encoding (Fitted on Train ONLY) ---
    global_mean = train['demand'].mean()
    agg = train.groupby('geohash')['demand'].agg(['count', 'mean']).reset_index()
    smoothing = 10
    agg['smoothed_demand'] = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    agg = agg[['geohash', 'smoothed_demand']]
    
    # Combine train and test to apply all other features consistently
    combined = pd.concat([train, test], sort=False).reset_index(drop=True)
    
    # Merge the smoothed demand
    combined = pd.merge(combined, agg, on='geohash', how='left')
    combined['smoothed_demand'] = combined['smoothed_demand'].fillna(global_mean)
    
    # --- 2. Cyclical Time Features & Rush Hour ---
    combined['hour'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[0]) if pd.notnull(x) else 0)
    combined['minute'] = combined['timestamp'].apply(lambda x: int(str(x).split(':')[1]) if pd.notnull(x) else 0)
    
    combined['hour_sin'] = np.sin(2 * np.pi * combined['hour'] / 24.0)
    combined['hour_cos'] = np.cos(2 * np.pi * combined['hour'] / 24.0)
    combined['minute_sin'] = np.sin(2 * np.pi * combined['minute'] / 60.0)
    combined['minute_cos'] = np.cos(2 * np.pi * combined['minute'] / 60.0)
    
    combined['is_rush_hour'] = combined['hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)
    combined['day_of_week'] = combined['day'] % 7
    combined['is_weekend'] = combined['day_of_week'].apply(lambda x: 1 if x in [5, 6] else 0)
    
    # The "Perfect Storm" Feature Cross
    combined['geo_rush_cross'] = combined['geohash'].astype(str) + "_" + combined['is_rush_hour'].astype(str)
    
    combined.drop(['timestamp', 'hour', 'minute'], axis=1, inplace=True)
    
    print("Cleaning missing values...")
    # --- 3. Missing Values ---
    combined['RoadType'] = combined['RoadType'].fillna('Unknown')
    combined['Weather'] = combined['Weather'].fillna('Unknown')
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].mode()[0])
    combined['LargeVehicles'] = combined['LargeVehicles'].fillna('Unknown')
    combined['Landmarks'] = combined['Landmarks'].fillna('Unknown')
    
    print("Encoding categories...")
    # --- 4. Encoding ---
    cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_rush_cross']
    for col in cat_cols:
        combined[col] = combined[col].astype(str)
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col])
        
    # Split exactly back into Train and Test
    train_processed = combined.iloc[:train_len].copy()
    test_processed = combined.iloc[train_len:].copy()
    
    X_train = train_processed.drop(['Index', 'demand'], axis=1)
    y_train = train_processed['demand']
    X_test = test_processed.drop(['Index', 'demand'], axis=1)
    
    return X_train, y_train, X_test, test_indices

print("Function Loaded! Ready to process data.")

Function Loaded! Ready to process data.


In [3]:
# 1. Process the data
X_train, y_train, X_test, test_indices = generate_ultimate_submission()

# 2. Load the 100-Trial Optuna Parameters
lgb_params = {    
    'n_estimators': 500,    
    'learning_rate': 0.15557591569719165,    
    'num_leaves': 85,    
    'max_depth': 7,    
    'min_child_samples': 13,    
    'subsample': 0.7339498255865238,    
    'colsample_bytree': 0.9772210751245085,    
    'objective': 'regression',    
    'random_state': 42,    
    'verbosity': -1
}

xgb_params = {    
    'n_estimators': 363,    
    'learning_rate': 0.09215538167220788,    
    'max_depth': 10,
    'objective': 'reg:squarederror',
    'random_state': 42,    
    'n_jobs': -1
}

# 3. Create the models
model_lgb = lgb.LGBMRegressor(**lgb_params)
model_xgb = xgb.XGBRegressor(**xgb_params)

# 4. Combine them into the Voting Committee
ensemble_model = VotingRegressor([
    ('lightgbm', model_lgb),
    ('xgboost', model_xgb)
])

# 5. Apply Peak-Demand Weights (Forcing it to respect the 1.0 traffic spikes)
print("\nApplying Peak-Demand Weights...")
train_weights = np.where(y_train > 0.8, 3.0, 1.0)

# 6. Train on 100% of the data!
print("Training Final Master Ensemble on 100% of data... (This may take a minute)")
ensemble_model.fit(X_train, y_train, sample_weight=train_weights)
print("Training Complete!")

Loading Train and Test datasets...
Applying Winning Feature Engineering...
Cleaning missing values...
Encoding categories...

Applying Peak-Demand Weights...
Training Final Master Ensemble on 100% of data... (This may take a minute)


  File "c:\Users\tanay\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
  File "c:\Users\tanay\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\tanay\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 556, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\tanay\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
            

Training Complete!


In [4]:
print("Generating Predictions for test.csv...")
predictions = ensemble_model.predict(X_test)

# Assemble the final submission format
submission = pd.DataFrame({
    'Index': test_indices,
    'demand': np.clip(predictions, a_min=0, a_max=None) # Hard floor at 0
})

# Display the first few rows so you can see your final masterpiece
display(submission.head())

# Save directly to your dataset folder
output_path = 'C:/Users/tanay/Documents/gridlock/e88186124ec611f1/dataset/ULTIMATE_submission.csv'
submission.to_csv(output_path, index=False)

print(f"\n🎉 SUCCESS! Your final, highly-optimized predictions are ready.")
print(f"Saved to: {output_path}")

Generating Predictions for test.csv...


,Index,demand
0,0,0.075919
1,1,0.019277
2,2,0.028897
3,3,0.023543
4,4,0.050529



🎉 SUCCESS! Your final, highly-optimized predictions are ready.
Saved to: C:/Users/tanay/Documents/gridlock/e88186124ec611f1/dataset/ULTIMATE_submission.csv


In [5]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# --- Example Data ---
# y_true = The actual real-world demand (from your validation split)
# y_pred = What your model guessed
y_true = [0.85, 0.10, 0.50, 1.00, 0.35]
y_pred = [0.80, 0.12, 0.45, 0.95, 0.35]

# --- 1. R-Squared (Accuracy/Variance Explained) ---
# Higher is better (1.0 is perfect)
r2 = r2_score(y_true, y_pred)

# --- 2. Mean Absolute Error (MAE) ---
# Lower is better (0.0 is perfect). Measures the average mistake in the exact units of your data.
mae = mean_absolute_error(y_true, y_pred)

# --- 3. Mean Squared Error (MSE) ---
# Lower is better. Punishes large mistakes heavily by squaring the errors.
mse = mean_squared_error(y_true, y_pred)

# --- 4. Root Mean Squared Error (RMSE) ---
# Lower is better. The square root of MSE, bringing the error back to normal units.
rmse = np.sqrt(mse) 

# --- Print the Results ---
print("--- MODEL EVALUATION METRICS ---")
print(f"R-Squared (R2):       {r2 * 100:.2f}%")
print(f"Mean Absolute Error:  {mae:.4f}")
print(f"Mean Squared Error:   {mse:.4f}")
print(f"Root Mean Sq Error:   {rmse:.4f}")
print("--------------------------------")

--- MODEL EVALUATION METRICS ---
R-Squared (R2):       98.53%
Mean Absolute Error:  0.0340
Mean Squared Error:   0.0016
Root Mean Sq Error:   0.0397
--------------------------------
